# Raw to Bronze
Pulls commits, issues and pull requests from the GitHub API.
Stores the exact raw record as text, plus a few tag columns.
Uses MERGE INTO, so running this twice never makes duplicate rows.

### How this one notebook covers full load, incremental load, and backfill
All three are just "give me records in a time window."
- **Full load**: since_date = 12 months ago, until_date = blank.
- **Incremental load**: since_date = the last time you ran this, until_date = blank.
- **Backfill**: since_date and until_date = any past window you want to redo.

Nothing is hard-coded. Every date, every repo, comes from a widget.

In [0]:
%run ./00_setup_and_logging

# Setup and logging
This notebook does two jobs:
1. Makes the schema (database) and all tables, if they don't exist yet.
2. Gives small helper functions that write one row to the log table every time a step runs.

Run this notebook by itself once. Other notebooks call it with `%run`.

Using workspace.dav_project


## Bronze tables
One table per entity (commits, issues, pulls), plus one quarantine table for
records that could not even be given a key. Every table has `load_timestamp`.

ok: bronze_commits
ok: bronze_issues
ok: bronze_pulls


ok: bronze_quarantine


...

ok: pipeline_execution_logs


## Phase 3 upgrade of the log table
The log table already exists from Phase 2, so `CREATE TABLE IF NOT EXISTS`
above does not change it. This cell adds the two new columns to an old
table, only if they are missing. Safe to run any number of times.
- `rows_aggregated`: how many source rows were read and rolled up (Silver to Gold).
- `duration_seconds`: how long the step took.

column already there: rows_aggregated
column already there: duration_seconds


## Small helper functions
`new_run_id()` makes one ID for one notebook run.
`log_run(...)` writes one row to the log table. Call it even when a step fails.
`get_merge_metrics(...)` reads how many rows a MERGE just inserted and updated.

helpers ready: new_run_id, now_utc, log_run, get_merge_metrics


In [0]:
dbutils.widgets.text("repos",
    "pandas-dev/pandas,numpy/numpy,scikit-learn/scikit-learn,matplotlib/matplotlib,"
    "scipy/scipy,fastapi/fastapi,pallets/flask,pytest-dev/pytest",
    "Repos (comma separated)")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental", "backfill"], "Load type")
dbutils.widgets.text("since_date", "", "Since date (YYYY-MM-DDTHH:MM:SSZ, blank = 24h ago)")
dbutils.widgets.text("until_date", "", "Until date (blank = no upper bound)")
dbutils.widgets.text("batch_id", "", "Batch ID (blank = auto)")
dbutils.widgets.text("github_token", "", "GitHub token (paste for this run only, clear before saving)")

REPOS = [r.strip() for r in dbutils.widgets.get("repos").split(",") if r.strip()]
LOAD_TYPE = dbutils.widgets.get("load_type")
SINCE_DATE = dbutils.widgets.get("since_date").strip() or None
UNTIL_DATE = dbutils.widgets.get("until_date").strip() or None
GITHUB_TOKEN = dbutils.widgets.get("github_token").strip() or None
BATCH_ID_OVERRIDE = dbutils.widgets.get("batch_id").strip() or None

print("repos:", REPOS)
print("load_type:", LOAD_TYPE, "| since:", SINCE_DATE, "| until:", UNTIL_DATE)

repos: ['pandas-dev/pandas', 'numpy/numpy', 'scikit-learn/scikit-learn', 'matplotlib/matplotlib', 'scipy/scipy', 'fastapi/fastapi', 'pallets/flask', 'pytest-dev/pytest']
load_type: incremental | since: None | until: None


In [0]:
import json
import time
import urllib.request
import urllib.error
import urllib.parse
from datetime import datetime, timedelta, timezone
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

if SINCE_DATE is None and LOAD_TYPE == "incremental":
    SINCE_DATE = (now_utc() - timedelta(days=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
    print("since_date was blank, using 24h ago:", SINCE_DATE)
if SINCE_DATE is None and LOAD_TYPE == "full":
    SINCE_DATE = (now_utc() - timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%SZ")
    print("since_date was blank, using 12 months ago:", SINCE_DATE)

RUN_ID = new_run_id()
BATCH_ID = BATCH_ID_OVERRIDE or f"{LOAD_TYPE}_{(SINCE_DATE or 'all').replace(':','').replace('-','')}"
print("run_id:", RUN_ID, "| batch_id:", BATCH_ID)

since_date was blank, using 24h ago: 2026-10-23T14:32:45Z
run_id: 196bd581-0948-4b31-9d28-cdb1d3ce2546 | batch_id: incremental_20261023T143245Z


## Explicit Bronze schema
Same shape for all three entities. This is the only schema the writer uses.
No `inferSchema`, anywhere.

In [0]:
BRONZE_SCHEMA = StructType([
    StructField("record_key", StringType(), False),
    StructField("source_repo", StringType(), False),
    StructField("entity_type", StringType(), False),
    StructField("raw_json", StringType(), False),
    StructField("api_params", StringType(), True),
    StructField("load_type", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("load_timestamp", TimestampType(), False),
])

QUARANTINE_SCHEMA = StructType([
    StructField("source_repo", StringType(), True),
    StructField("entity_type", StringType(), True),
    StructField("raw_record", StringType(), True),
    StructField("reason", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("load_timestamp", TimestampType(), True),
])

## Fetch one repo, one entity, one time window
Commits use the API's own since/until. Issues and pull requests don't
support `until` on the server, so we sort newest-first and filter here,
stopping early once we page past the `since` floor.

In [0]:
PER_PAGE = 100


def _headers():
    h = {"Accept": "application/vnd.github+json"}
    if GITHUB_TOKEN:
        h["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return h


MAX_RETRIES = 4  # for short GitHub hiccups (500/502/503/504, timeouts)


def _get(url, params):
    full = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(full, headers=_headers())
    attempt = 0
    while True:
        try:
            with urllib.request.urlopen(req, timeout=30) as resp:
                return resp.status, json.loads(resp.read().decode("utf-8")), dict(resp.headers)
        except urllib.error.HTTPError as e:
            if e.code in (403, 429) and e.headers.get("X-RateLimit-Remaining") == "0":
                reset = int(e.headers.get("X-RateLimit-Reset", time.time() + 30))
                wait = max(reset - time.time(), 1)
                print(f"  rate limited, sleeping {int(wait)}s")
                time.sleep(wait + 1)
                continue
            if e.code in (500, 502, 503, 504) and attempt < MAX_RETRIES:
                attempt += 1
                print(f"  GitHub error {e.code}, retry {attempt}/{MAX_RETRIES}")
                time.sleep(2 ** attempt)
                continue
            raise
        except (urllib.error.URLError, TimeoutError) as e:
            if attempt < MAX_RETRIES:
                attempt += 1
                print(f"  network error ({e}), retry {attempt}/{MAX_RETRIES}")
                time.sleep(2 ** attempt)
                continue
            raise


def fetch_window(repo, entity, since_date, until_date):
    """Returns (records, params_used). records is a list of (key, raw_dict, reason_if_bad)."""
    base = f"https://api.github.com/repos/{repo}"
    records = []

    if entity == "commits":
        params = {"per_page": PER_PAGE}
        if since_date:
            params["since"] = since_date
        if until_date:
            params["until"] = until_date
        page = 1
        while True:
            params["page"] = page
            _, batch, _ = _get(f"{base}/commits", params)
            if not batch:
                break
            for rec in batch:
                key = rec.get("sha")
                records.append((key, rec, None if key else "missing sha"))
            if len(batch) < PER_PAGE:
                break
            page += 1
        return records, params

    # issues and pulls: no server-side 'until', filter here, newest first
    endpoint = "issues" if entity == "issues" else "pulls"
    params = {"per_page": PER_PAGE, "state": "all", "sort": "updated", "direction": "desc"}
    if entity == "issues" and since_date:
        params["since"] = since_date  # server-side lower bound as a speed-up; we still re-check it below
    page = 1
    while True:
        params["page"] = page
        _, batch, _ = _get(f"{base}/{endpoint}", params)
        if not batch:
            break
        reached_floor = False
        for rec in batch:
            updated_at = rec.get("updated_at")
            if since_date and updated_at and updated_at < since_date:
                reached_floor = True
                continue
            if until_date and updated_at and updated_at > until_date:
                continue
            key = str(rec.get("id")) if rec.get("id") is not None else None
            records.append((key, rec, None if key else "missing id"))
        if reached_floor or len(batch) < PER_PAGE:
            break
        page += 1
    return records, params

## Run it: one entity at a time, across all repos
Each entity gets its own log row. One repo failing does not stop the others.

In [0]:
def load_entity_to_bronze(entity):
    start = now_utc()
    good_rows, bad_rows = [], []
    params_by_repo = {}
    failed_repos = []
    try:
        for repo in REPOS:
            try:
                records, params = fetch_window(repo, entity, SINCE_DATE, UNTIL_DATE)
                params_by_repo[repo] = params
            except Exception as e:
                print(f"  {repo}/{entity} failed to fetch: {e}")
                failed_repos.append(f"{repo} ({e})")
                continue
            for key, rec, reason in records:
                if reason:
                    bad_rows.append((repo, entity, json.dumps(rec), reason, BATCH_ID, RUN_ID, now_utc()))
                else:
                    good_rows.append((
                        key, repo, entity, json.dumps(rec),
                        json.dumps(params_by_repo[repo]), LOAD_TYPE, BATCH_ID, RUN_ID, now_utc(),
                    ))
            print(f"  {repo}/{entity}: {len(records)} records in window")

        # GitHub can return the same record on two pages when the list shifts
        # while we are paging (busy repos). MERGE refuses duplicate keys in one
        # batch, so keep one row per record_key (the last one seen).
        before = len(good_rows)
        good_rows = list({row[0]: row for row in good_rows}.values())
        if len(good_rows) < before:
            print(f"  removed {before - len(good_rows)} repeated records inside this batch")

        inserted = updated = 0
        if good_rows:
            df = spark.createDataFrame(good_rows, schema=BRONZE_SCHEMA)
            df.createOrReplaceTempView("bronze_batch_view")
            full_table = f"{CATALOG}.{SCHEMA}.bronze_{entity}"
            spark.sql(f"""
                MERGE INTO {full_table} AS t
                USING bronze_batch_view AS s
                ON t.record_key = s.record_key
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            m = get_merge_metrics(full_table)
            inserted, updated = m["inserted"], m["updated"]

        if bad_rows:
            qdf = spark.createDataFrame(bad_rows, schema=QUARANTINE_SCHEMA)
            qdf.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_quarantine")

        # If any repo could not be fetched, the batch is incomplete. The rows we
        # did get are saved, but the run is logged as FAILED so nobody thinks
        # the window is complete.
        status = "FAILED" if failed_repos else "SUCCESS"
        err = ("fetch failed for: " + "; ".join(failed_repos)) if failed_repos else None
        log_run(RUN_ID, "Raw-to-Bronze", entity,
                {"repos": REPOS, "load_type": LOAD_TYPE, "since": SINCE_DATE,
                 "until": UNTIL_DATE, "batch_id": BATCH_ID},
                start, now_utc(), status,
                rows_inserted=inserted, rows_updated=updated, rows_quarantined=len(bad_rows),
                error_message=err)
        if failed_repos:
            print(f"INCOMPLETE: {entity}: {err}")
        return inserted, updated, len(bad_rows), status

    except Exception as e:
        log_run(RUN_ID, "Raw-to-Bronze", entity,
                {"repos": REPOS, "load_type": LOAD_TYPE, "since": SINCE_DATE,
                 "until": UNTIL_DATE, "batch_id": BATCH_ID},
                start, now_utc(), "FAILED", error_message=str(e))
        print(f"FAILED: {entity}: {e}")
        return 0, 0, 0, "FAILED"


results = {}
for entity in ["commits", "issues", "pulls"]:
    print(f"\n=== {entity} ===")
    results[entity] = load_entity_to_bronze(entity)

print("\nSummary (inserted, updated, quarantined, status):")
for entity, r in results.items():
    print(f"  {entity}: {r}")


=== commits ===
  pandas-dev/pandas/commits: 5 records in window
  numpy/numpy/commits: 4 records in window
  scikit-learn/scikit-learn/commits: 0 records in window
  matplotlib/matplotlib/commits: 2 records in window
  scipy/scipy/commits: 4 records in window
  fastapi/fastapi/commits: 3 records in window
  pallets/flask/commits: 0 records in window
  pytest-dev/pytest/commits: 0 records in window
[log] Raw-to-Bronze / commits -> SUCCESS (inserted=0, updated=18, quarantined=0, aggregated=0, seconds=9.9)

=== issues ===
  pandas-dev/pandas/issues: 87 records in window
  numpy/numpy/issues: 30 records in window
  scikit-learn/scikit-learn/issues: 20 records in window
  matplotlib/matplotlib/issues: 14 records in window
  scipy/scipy/issues: 35 records in window
  fastapi/fastapi/issues: 9 records in window
  pallets/flask/issues: 1 records in window
  pytest-dev/pytest/issues: 12 records in window
[log] Raw-to-Bronze / issues -> SUCCESS (inserted=0, updated=208, quarantined=0, aggregat

## Check the proof
Run this cell. Then run the whole notebook again with the same batch_id.
Row counts in the bronze tables should not change the second time.

In [0]:
display(spark.sql(f"""
    SELECT run_id, layer, entity, status, rows_inserted, rows_updated, rows_quarantined, start_time
    FROM {CATALOG}.{SCHEMA}.pipeline_execution_logs
    WHERE run_id = '{RUN_ID}'
    ORDER BY start_time
"""))

run_id,layer,entity,status,rows_inserted,rows_updated,rows_quarantined,start_time
196bd581-0948-4b31-9d28-cdb1d3ce2546,Raw-to-Bronze,commits,SUCCESS,0,18,0,2026-10-24T14:32:45.931Z
196bd581-0948-4b31-9d28-cdb1d3ce2546,Raw-to-Bronze,issues,SUCCESS,0,208,0,2026-10-24T14:32:57.460Z
196bd581-0948-4b31-9d28-cdb1d3ce2546,Raw-to-Bronze,pulls,SUCCESS,1,147,0,2026-10-24T14:33:12.062Z


## Hand the result back
When another notebook (`05_run_pipeline`) runs this one, this is what it
reads. It says which batch was loaded and whether every step worked.

In [0]:
all_ok = all(r[3] == "SUCCESS" for r in results.values())
dbutils.notebook.exit(json.dumps({
    "batch_id": BATCH_ID,
    "run_id": RUN_ID,
    "all_ok": all_ok,
    "results": {k: {"inserted": v[0], "updated": v[1], "quarantined": v[2], "status": v[3]}
                for k, v in results.items()},
}))